# 🤖 Agentic RAG Workshop — 4 hoursเดียวจบ

**Agentic RAG: From Zero to Hero** | Workshop 4 ชม.

---

### 🎯 จุดประสงค์การเรียนรู้

เมื่อจบ workshop ผู้เรียนสามารถ:
1. **เตรียมข้อมูลสำหรับ RAG** — Chunking + Embedding + VectorDB
2. **สร้าง AI Agent** — ใช้ Google ADK สร้าง Agent + Custom Tool
3. **สร้าง Agentic RAG** ⭐ — Agent ที่ค้นหาจาก VectorDB + ตัดสินใจ + ตอบคำถามเอง
4. **วัดคุณภาพ** — ใช้ LLM-as-Judge ประเมิน Agent

### ⏱️ Timeline

| Part | Duration | เนื้อหา |
|------|:----:|--------|
| 📢 Part 1 | 1 ชม. 20 นาที | Data → Chunk → Embed → Qdrant → Retrieve |
| ☕ พัก | 10 นาที | |
| 📢 Part 2 | 1 ชม. 30 นาที | Agent → Tool → RAG Agent → Agentic RAG |
| 🧪 Part 3 | 1 ชม. | Exercise + Q&A |

---
## 📢 Part 1: Data → VectorDB

### RAG ?

**R**etrieval-**A**ugmented **G**eneration = + 

```
LLM 
→ "" 
→ LLM "" 
```

| Pipeline |
|---|
| 📄 Text → ✂️ Chunking → 🔢 Embedding → 🗄️ Qdrant → 🔎 Retrieve → 🤖 Agent → 💬 Answer |

## 📦 Section 0: Setup

 libraries 

In [ ]:
%%time
# libraries
!pip install -q google-adk \
 google-genai \
 sentence-transformers \
 qdrant-client \
 langchain-text-splitters \
 scikit-learn

print('✅ !')
print('⚠️ install Restart runtime: Runtime → Restart session → Run cell ')

In [ ]:
%%time
# Gemini API Key
import os

try:
 from google.colab import userdata
 os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
 print('✅ API Key Colab Secrets')
except Exception:
 os.environ['GOOGLE_API_KEY'] = input('🔑 Gemini API Key: ')

# API
from google import genai
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
resp = client.models.generate_content(model='gemini-2.5-flash', contents=' 1 ')
print(f'🤖 Gemini: {resp.text}')

### 📄 Prepare sample data

สร้างThai case-study data used throughout the workshop

In [ ]:
%%time
# Create sample data
import os
os.makedirs('data', exist_ok=True)

sample_texts = {
    'case_ict_mahidol.txt': '''กรณีศึกษา: AI ที่คณะ ICT มหาวิทยาลัยมหิดล

คณะเทคโนโลยีสารสนเทศและการสื่อสาร (ICT) มหาวิทยาลัยมหิดล
เป็นที่ตั้งของ Mahidol AI Center สถาบันปัญญาประดิษฐ์มหิดล
มีคลัสเตอร์วิจัย MIKE (Machine Intelligence and Knowledge Engineering)

มุ่งเน้นงานวิจัย AI ด้านการแพทย์ จีโนมิกส์ และ Federated Learning
โครงการเด่น: Siribot แชทบอทสำหรับโรงพยาบาลศิริราช
AI Medical Imaging วิเคราะห์ภาพ X-Ray และ VIS4ION ระบบนำทางผู้พิการทางสายตา

คณะ ICT ยังนำ RAG มาสร้างระบบ AI Tutor
ช่วยตอบคำถามวิชาเรียนจากเอกสารประกอบการสอน
นักศึกษาถามคำถามได้ตลอด 24 hours โดยค้นคำตอบจาก lecture notes slides และตำราเรียน''',

    'case_healthcare.txt': '''กรณีศึกษา: AI ในการแพทย์ไทย

โรงพยาบาลศิริราชนำ AI มาวิเคราะห์ภาพถ่ายทางการแพทย์
เช่น การตรวจจับมะเร็งปอดจากภาพ X-ray ด้วย Deep Learning
ผลการทดสอบพบว่า AI มีความแม่นยำ 95%

มีการใช้ NLP วิเคราะห์เวชระเบียนอิเล็กทรอนิกส์
ลดDurationการอ่านเวชระเบียนจาก 15 นาทีเหลือ 2 นาทีต่อเคส

ความท้าทาย: ข้อมูลทางการแพทย์ภาษาไทยมีจำนวนจำกัด
ต้องใช้ Transfer Learning จากโมเดลภาษาอังกฤษ''',

    'case_banking.txt': '''กรณีศึกษา: AI ในธนาคารและการเงิน

ธนาคารกสิกรไทยพัฒนาระบบ Chatbot ที่ใช้ LLM ร่วมกับ RAG
ตอบคำถามลูกค้าเกี่ยวกับผลิตภัณฑ์ทางการเงิน

ระบบ RAG ค้นหาข้อมูลจากฐานความรู้ภายใน
ประกอบด้วยเอกสารผลิตภัณฑ์ เงื่อนไขบริการ และ FAQ
LLM สร้างคำตอบที่เป็นธรรมชาติจากข้อมูลที่ค้นพบ

ผลลัพธ์: ลดภาระ Call Center 40%
ความพึงพอใจลูกค้าเพิ่มขึ้น 25%
ใช้ Vector Database เก็บ Embedding + Hybrid Search''',

    'case_agriculture.txt': '''กรณีศึกษา: AI ในการเกษตรอัจฉริยะ

Smart Farming ในไทยใช้ AI วิเคราะห์ภาพถ่ายดาวเทียม
และข้อมูลจาก IoT Sensor เพื่อพยากรณ์ผลผลิตและเฝ้าระวังโรคพืช

ระบบ Computer Vision ตรวจจับโรคข้าวจากภาพถ่ายใบข้าว
ใช้ CNN ที่ Train ด้วยภาพถ่ายโรคข้าวกว่า 50,000 ภาพ

มีการใช้ NLP วิเคราะห์ข้อมูลราคาสินค้าเกษตร
จากข่าวสาร รายงานภาครัฐ และ Social Media
ช่วยเกษตรกรตัดสินใจเรื่องการปลูกและการขาย'''
}

for fname, content in sample_texts.items():
    with open(f'data/{fname}', 'w', encoding='utf-8') as f:
        f.write(content)

print(f'✅ สร้างไฟล์ตัวอย่าง {len(sample_texts)} ไฟล์')
for fname in sorted(sample_texts.keys()):
    print(f'  📄 {fname}')

---
## ✂️ Section 1: Chunking — 

### Chunk?

- LLM **context window ** — 100 
- Embedding model ****
- Chunk **retrieval **

| | | | |
|------|---------|-------|--------|
| **Fixed-size** | | , | |
| **Recursive** ⭐ | separator | , | separators |

In [ ]:
%%time
# โหลดข้อความจากไฟล์
all_text = ''
for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath) and fname.endswith('.txt'):
        with open(fpath, 'r', encoding='utf-8') as f:
            all_text += '\n\n' + f.read()

print(f'📄 ข้อความรวม: {len(all_text)} ตัวอักษร')

In [ ]:
%%time
# ─── Fixed-size Chunking ───
def fixed_chunk(text, size=200, overlap=30):
 chunks = []
 for i in range(0, len(text), size - overlap):
 chunks.append(text[i:i+size])
 return chunks

fixed_chunks = fixed_chunk(all_text, size=200, overlap=30)
print(f'📊 Fixed-size: {len(fixed_chunks)} chunks')
print(f'\n--- Chunk 1 (: ) ---')
print(fixed_chunks[0][:150] + '...')

In [ ]:
%%time
# ─── Recursive Chunking ⭐ () ───
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
 chunk_size=200,
 chunk_overlap=30,
 separators=['\n\n', '\n', ' ', ''] # : → → → 
)

chunks = splitter.split_text(all_text)
print(f'📊 Recursive: {len(chunks)} chunks')
print(f'\n--- Chunk 1 (: ) ---')
print(chunks[0])
print(f'\n--- Chunk ---')
print(chunks[-1])

### 💡 
- **Fixed-size** /
- **Recursive** `\n\n` `\n` → 
- `chunk_size` → context
- `chunk_size` → context noise

> 🎯 workshop **Recursive Chunking** 

---
## 🔢 Section 2: Embedding — Vector

### Embedding ?

 → (Vector) 

```
"" → [0.21, -0.85, 0.44, ...] (1024 )
"" → [0.19, -0.82, 0.41, ...] ← "" ()
"" → [-0.55, 0.31, -0.12, ...] ← "" ()
```

**Model :** `intfloat/multilingual-e5-large` — 

> ⚠️ prefix: `'query: '` , `'passage: '` 

In [ ]:
%%time
# Embedding Model ( ~2 )
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
print(f'✅ model | Vector size: {embed_model.get_sentence_embedding_dimension()}')

In [ ]:
%%time
# embedding chunk
passages = ['passage: ' + c for c in chunks]
chunk_embeddings = embed_model.encode(passages, show_progress_bar=True)

print(f'✅ Embed {len(chunks)} chunks ')
print(f'📐 Shape: {chunk_embeddings.shape}')

In [ ]:
%%time
# Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = 'query: AI '
query_emb = embed_model.encode(query)

# similarity chunk
scores = cosine_similarity([query_emb], chunk_embeddings)[0]
top_idx = np.argsort(scores)[::-1][:3]

print(f'🔍 Query: "{query}"')
print(f'\n📊 Top-3 chunks :')
for rank, idx in enumerate(top_idx, 1):
 print(f' {rank}. [score={scores[idx]:.4f}] {chunks[idx][:80]}...')

### 💡 
- **** → cosine similarity **** ( 1.0)
- prefix `query:` / `passage:` model 
- Embedding **** keyword 

> 🎯 embedding **Vector Database** 

---
## 🗄️ Section 3: Qdrant — Vector Database + Retrieve

### DB ?

- DB (SQL): keyword / exact match
- VectorDB: **** (semantic search)

**Qdrant** = VectorDB , , in-memory

In [ ]:
%%time
# Qdrant Client + Collection
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(':memory:') # RAM ( server)

COLLECTION = 'workshop_docs'
VECTOR_SIZE = chunk_embeddings.shape[1] # 1024

qdrant.create_collection(
 collection_name=COLLECTION,
 vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

print(f'✅ Collection "{COLLECTION}" (vector size={VECTOR_SIZE})')

In [ ]:
%%time
# Upsert chunk Qdrant
points = [
 models.PointStruct(
 id=i,
 vector=chunk_embeddings[i].tolist(),
 payload={'text': chunks[i], 'chunk_id': i}
 )
 for i in range(len(chunks))
]

qdrant.upsert(collection_name=COLLECTION, points=points)
print(f'✅ Upsert {len(points)} chunks Qdrant !')

In [ ]:
%%time
# ─── Search from Qdrant ───
def search_qdrant(query_text, top_k=3):
    """Search chunks from Qdrant using a query"""
    q_emb = embed_model.encode(f'query: {query_text}')
    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_emb.tolist(),
        limit=top_k
    ).points
    return results

# ทดลองค้นหา
results = search_qdrant('Mahidol AI Center งานวิจัยปัญญาประดิษฐ์')
print('🔍 Query: "Mahidol AI Center งานวิจัยปัญญาประดิษฐ์"')
print(f'\n📊 Top-{len(results)} ผลลัพธ์จาก Qdrant:')
for i, r in enumerate(results, 1):
    print(f'  {i}. [score={r.score:.4f}] {r.payload["text"][:80]}...')

### 💡 Observation
- Qdrant ค้นหาด้วย **vector similarity** → เข้าใจความหมาย ไม่ใช่แค่ keyword
- ถามเรื่อง "แพทย์" → ได้ผลเกี่ยวกับ case_healthcare.txt
- `top_k` ควบคุมจำนวนผลลัพธ์ที่ส่งให้ LLM

> 🎯 ตอนนี้เรามี **RAG Pipeline ครบแล้ว!** ต่อไปเราจะสร้าง **Agent** ที่ใช้ pipeline นี้

---
### ☕ พัก 10 นาที

---

## 📢 Part 2: Agent + Agentic RAG

---
## 🤖 Section 4: Your first Agent — Google ADK

### Agent vs Chatbot

🤔 **Think of it as:** คุณอยากจองตั๋วเครื่องบิน
- **Chatbot**: "กรุณาเลือก 1. ดูเที่ยวบิน 2. เช็คราคา 3. ติดต่อเจ้าหน้าที่" ← ทำได้แค่ตาม script
- **Agent**: "หาเที่ยวบิน กรุงเทพ→เชียงใหม่ วันศุกร์ ราคาถูกสุด จองเลย" ← **คิดเอง + ใช้เครื่องมือเอง**

| | Chatbot | Agent |
|---|---------|-------|
| ทำงาน | ถาม-ตอบตาม script | **คิด → ตัดสินใจ → ใช้เครื่องมือ** |
| ความยืดหยุ่น | ต่ำ (ต้อง hard-code) | สูง (ปรับตัวได้) |
| ตัวอย่าง | FAQ bot, เมนูโทรศัพท์ | AI จองตั๋ว, AI ค้นข้อมูล, Agentic RAG |

**Google ADK** = Agent Development Kit สร้าง Agent ด้วย Python ไม่กี่บรรทัด

In [ ]:
%%time
# Agent 
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# Agent ""
teacher_agent = LlmAgent(
 name='teacher_assistant',
 model='gemini-2.5-flash',
 instruction=' AI 3 '
)

print(f'✅ Agent "{teacher_agent.name}" ')

In [ ]:
# ─── คุยกับ Agent ───
import asyncio

async def chat_with_agent(agent, message):
    """ส่งข้อความไปหา Agent แล้วรับคำตอบ"""
    runner = InMemoryRunner(agent=agent, app_name='workshop')
    session = await runner.session_service.create_session(
        app_name='workshop', user_id='student'
    )
    
    content = genai_types.Content(
        role='user',
        parts=[genai_types.Part(text=message)]
    )
    
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    
    return response_text

# ถามคำถาม
answer = await chat_with_agent(teacher_agent, 'RAG คืออะไร?')
print(f'❓ คำถาม: RAG คืออะไร?')
print(f'🤖 Agent: {answer}')

### 💡 Observation
- Agent ตอบตาม `instruction` ที่เรากำหนด
- แต่ Agent ยังตอบจาก **ความรู้ทั่วไปของ LLM** เท่านั้น
- ยังไม่ได้ค้นจาก **ข้อมูลของเรา** → ต้องให้ "เครื่องมือ" (Tool)

> 🎯 ต่อไปเราจะสร้าง Tool ให้ Agent ใช้งาน

---
## 🔧 Section 5: Tool — Agent 

### FunctionTool ?

 Python function → Agent **** 

> ⚠️ **docstring !** Agent docstring tool 

In [ ]:
# Custom Tool
from google.adk.tools import FunctionTool

def calculate_bmi(weight_kg: float, height_cm: float) -> str:
 """ BMI () ()
 
 Args:
 weight_kg: 
 height_cm: 
 """
 height_m = height_cm / 100
 bmi = weight_kg / (height_m ** 2)
 if bmi < 18.5: status = ''
 elif bmi < 25: status = ''
 elif bmi < 30: status = ''
 else: status = ''
 return f'BMI = {bmi:.1f} ({status})'

bmi_tool = FunctionTool(calculate_bmi)

# Agent Tool
health_agent = LlmAgent(
 name='health_assistant',
 model='gemini-2.5-flash',
 instruction=' tool calculate_bmi ',
 tools=[bmi_tool]
)

print('✅ Health Agent + BMI Tool ')

In [ ]:
# Agent + Tool
answer = await chat_with_agent(health_agent, ' 70 . 175 . BMI ?')
print(f'❓ : 70 . 175 .')
print(f'🤖 Agent: {answer}')

### 💡 
- Agent ** docstring** `calculate_bmi` 
- hard-code: Agent tool
- **docstring = Agent **

> 🎯 **RAG Tool** Qdrant!

---
## 🔍 Section 6: RAG Agent ⭐ — Agent + VectorDB

### Workshop !

 Agent + Qdrant Pipeline Part 1

```
 → Agent → search_knowledge
→ Qdrant → context → Agent 
```

In [ ]:
# RAG Tool — Qdrant
def search_knowledge(query: str) -> str:
 """ Case Study AI 
 AI 
 
 Args:
 query: 
 """
 results = search_qdrant(query, top_k=3)
 context = '\n\n'.join([f'[{i+1}] {r.payload["text"]}' for i, r in enumerate(results)])
 return f' {len(results)} :\n{context}'

rag_tool = FunctionTool(search_knowledge)

# RAG Agent
rag_agent = LlmAgent(
 name='rag_assistant',
 model='gemini-2.5-flash',
 instruction=""" AI Assistant AI 

:
1. tool search_knowledge 
2. 
3. 
4. """,
 tools=[rag_tool]
)

print('✅ RAG Agent !')

In [ ]:
# ─── RAG Agent ───
questions = [
 'AI ?',
 ' ICT AI ?',
 ' AI ?'
]

for q in questions:
 answer = await chat_with_agent(rag_agent, q)
 print(f'\n{"="*60}')
 print(f'❓ {q}')
 print(f'🤖 {answer}')

### 💡 Observation
- Agent **decides on its own** ว่าจะSearch from Qdrant
- Answers come from **real retrieved data**, not only LLM prior knowledge.
- ต่างจาก RAG ธรรมดา: **Agent เลือกได้** ว่าจะค้นหรือตอบเลย

> 🎯 นี่คือ **Agentic RAG** — ต่อไปเราจะเพิ่ม Memory ให้จำบทสนทนาได้!

---
## 🚀 Section 7: Agentic RAG + Memory ⭐

### Session Memory

 Agent **** → 

In [ ]:
# ─── Agentic RAG + Memory ───
from google.adk.sessions import InMemorySessionService

# RAG Agent session 
runner = InMemoryRunner(agent=rag_agent, app_name='agentic_rag')

async def chat_with_memory(runner, session_id, message):
 content = genai_types.Content(
 role='user',
 parts=[genai_types.Part(text=message)]
 )
 response_text = ''
 async for event in runner.run_async(
 user_id='student', session_id=session_id, new_message=content
 ):
 if event.content and event.content.parts:
 for part in event.content.parts:
 if part.text:
 response_text += part.text
 return response_text

# session
session = await runner.session_service.create_session(
 app_name='agentic_rag', user_id='student'
)

# 3 
conversations = [
 'AI ?',
 ' NLP ?', # — Agent 
 '?' # 
]

for q in conversations:
 answer = await chat_with_memory(runner, session.id, q)
 print(f'\n{"="*60}')
 print(f'❓ {q}')
 print(f'🤖 {answer}')

### 💡 
- 2 3 **** Agent 
- **Session Memory** — Agent 
- **Agentic RAG = Agent + RAG + Memory** → + + 

> 🎯 : Data Pipeline + Agent + Tool + RAG + Memory = **Agentic RAG !**

---
## 📋 : 

| Part | Section | |
|------|---------|--------|
| 1 | Chunking | Recursive Splitter |
| 1 | Embedding | Vector (multilingual-e5-large) |
| 1 | Qdrant | + Vector |
| 2 | Agent | Agent Google ADK |
| 2 | Tool | Agent FunctionTool |
| 2 | RAG Agent ⭐ | Agent VectorDB |
| 2 | Agentic RAG ⭐ | Agent + RAG + Memory |

### 🔑 Key Takeaways

1. **** — "Garbage in, garbage out"
2. **Agent ≠ Chatbot** — Agent + 
3. **Agentic RAG** = Agent 

---

### 🧪 (1 )

 **`agentic_rag_4hr_homework.ipynb`** (10 )

> _"The best way to learn AI is to build with AI."_